## Conditional Email
Sends an email when conditions regarding an available item are met

In [6]:
import duckdb
from dotenv import load_dotenv
import os

import polars as pl
from polars import col

from datetime import date
import smtplib
from email.message import EmailMessage

In [7]:
# Target price
target_price = 100

# DuckDB Connection
con = duckdb.connect('../burger.db')

# Environment
env_path = '../.env'
load_dotenv(dotenv_path=env_path)

# Environment Variables
email = os.getenv('EMAIL')
email_password = os.getenv('EMAIL_PASSWORD')
destination_email = os.getenv('DESTINATION_EMAIL')
smtp_server = os.getenv('EMAIL_SMTP_SERVER')

### Get Data from Silver

In [8]:
today = date.today()

df_silver = con.execute("""
    SELECT * FROM silver
    WHERE loaded_at = ?
""", [today]).pl()

con.close()

### Isolate records that meet profile

In [9]:
df_positive = df_silver.filter(col('total_cost') <= target_price).select(
    col('total_cost'),
    col('item_url')
)

### Send Email

In [10]:
if df_positive.height > 0:
    lines = [
        f"${row['total_cost']:.2f} - {row['item_url']}"
        for row in df_positive.iter_rows(named=True)
    ]

    msg = EmailMessage()
    msg["Subject"] = f"Burgerchu alert: {df_positive.height} item(s) at or under ${target_price}"
    msg["From"] = email
    msg["To"] = destination_email
    msg.set_content("\n".join(lines))

    with smtplib.SMTP(smtp_server, 587) as s:
        s.starttls()
        s.login(email, email_password)
        s.send_message(msg)


    print(f"Email sent - {df_positive.height} matching item(s).")
else:
    print("No items met the target price today - no email sent.")

Email sent - 1 matching item(s).
